# Day 28 — Python + SQL Pipeline: Automated Report
**Estimated time:** 2 hours  
**Learning objectives:**
1. Build a modular, production-quality analytics pipeline with type hints and docstrings
2. Chain: data load → DQ checks → KPI calculation → formatted report export
3. Design a simple CLI interface for parameterised report generation

---
> **SAP Context:** This pipeline mirrors what you'd build as a replacement for a manually-run BW workbook or a BI report that requires analyst intervention. The goal: zero-touch, parameterisable report that runs on a schedule. In enterprise settings, this would connect to an SFTP/S3 bucket for data and an email/SharePoint system for delivery.


## 0. Pipeline Architecture Overview

```
                ┌─────────────────────────────────────────┐
                │           analytics_pipeline.py          │
                │                                         │
                │  load_data()                            │
                │      └─► DataFrames from CSV            │
                │                                         │
                │  run_dq_checks(dfs)                     │
                │      └─► DQ scorecard DataFrame         │
                │                                         │
                │  calculate_kpis(dfs, plant, period)     │
                │      └─► KPI summary dict               │
                │                                         │
                │  format_report(kpis, dq, period)        │
                │      └─► CSV + HTML files               │
                └─────────────────────────────────────────┘

CLI: python report.py --plant 1000 --period 202412
```


## 1. Module: load_data()

In [ ]:
from pathlib import Path
from typing import Optional
import pandas as pd
import numpy as np
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('analytics_pipeline')

DATA = Path('../../data/raw')

def load_data(data_path: Path = DATA) -> dict[str, pd.DataFrame]:
    """Load all datasets from CSV files into a dictionary of DataFrames.

    Args:
        data_path: Path to the directory containing the CSV files.

    Returns:
        Dict mapping table name to DataFrame.

    Raises:
        FileNotFoundError: If a required CSV file is missing.
    """
    files = {
        'materials': 'materials_inventory.csv',
        'sales_orders': 'sales_orders.csv',
        'cost_centers': 'cost_center_actuals.csv',
        'hr': 'hr_headcount.csv',
        'bw_kpis': 'bw_sales_kpis.csv',
    }

    dfs: dict[str, pd.DataFrame] = {}
    for name, filename in files.items():
        path = data_path / filename
        if not path.exists():
            raise FileNotFoundError(f'Required data file not found: {path}')
        dfs[name] = pd.read_csv(path)
        logger.info('Loaded %s: %d rows, %d cols', name, len(dfs[name]), len(dfs[name].columns))

    return dfs

# Test it
dfs = load_data()
print('Tables loaded:', list(dfs.keys()))

## 2. Module: run_dq_checks()

In [ ]:
# YOUR SOLUTION

def run_dq_checks(dfs: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Run a suite of data quality checks on all loaded DataFrames.

    Args:
        dfs: Dict of DataFrames returned by load_data().

    Returns:
        DataFrame with columns: check_name, status, record_count,
        failure_count, failure_pct, notes
    """
    results = []

    def add(name: str, total: int, failures: int, notes: str = '') -> None:
        pct = round(failures / total * 100, 2) if total > 0 else 0.0
        if failures == 0:
            status = 'PASS'
        elif pct < 5.0:
            status = 'WARN'
        else:
            status = 'FAIL'
        results.append({
            'check_name': name, 'status': status,
            'record_count': total, 'failure_count': failures,
            'failure_pct': pct, 'notes': notes
        })
        logger.info('DQ %-45s %s (%d failures)', name, status, failures)

    so = dfs['sales_orders']
    mat = dfs['materials']
    hr = dfs['hr']
    cc = dfs['cost_centers']

    # Null checks
    add('NULL: sales_orders.NETWR', len(so), so['NETWR'].isna().sum(), 'Net value required')
    add('NULL: hr.SALARY', len(hr), hr['SALARY'].isna().sum(), 'Salary required')
    add('NULL: materials.STPRS', len(mat), mat['STPRS'].isna().sum(), 'Standard price required')

    # Duplicates
    dup_so = len(so) - so[['VBELN','POSNR']].drop_duplicates().shape[0]
    add('Duplicate PK: sales_orders VBELN+POSNR', len(so), dup_so)

    # Referential integrity: sales orders → materials
    orphans = so[~so['MATNR'].isin(mat['MATNR'])].shape[0]
    add('RI: sales_orders.MATNR in materials', len(so), orphans, 'Orphaned MATNR')

    # HR cost center in CO
    hr_kostl = set(hr['KOSTL'].dropna())
    cc_kostl = set(cc['KOSTL'].dropna())
    missing_cc = len(hr_kostl - cc_kostl)
    add('RI: hr.KOSTL in cost_centers', len(hr_kostl), missing_cc, 'HR cost centers not in CO')

    # Date sanity
    hr_parsed = pd.to_datetime(hr['HIRE_DATE'], errors='coerce')
    term_parsed = pd.to_datetime(hr['TERM_DATE'], errors='coerce')
    bad_dates = ((term_parsed.notna()) & (term_parsed < hr_parsed)).sum()
    add('Date: TERM_DATE >= HIRE_DATE', len(hr), int(bad_dates))

    return pd.DataFrame(results)

dq_scorecard = run_dq_checks(dfs)
print('\nDQ Scorecard:')
print(dq_scorecard[['check_name','status','failure_count','failure_pct']].to_string(index=False))

## 3. Module: calculate_kpis()

In [ ]:
# YOUR SOLUTION

def calculate_kpis(
    dfs: dict[str, pd.DataFrame],
    plant: Optional[str] = None,
    period: Optional[int] = None
) -> dict:
    """Calculate key business KPIs, optionally filtered by plant and period.

    Args:
        dfs: Dict of DataFrames from load_data().
        plant: SAP plant code filter (e.g., '1000'). None = all plants.
        period: BW period in YYYYMM format (e.g., 202412). None = all periods.

    Returns:
        Dict of KPI name → value.
    """
    bw = dfs['bw_kpis'].copy()
    so = dfs['sales_orders'].copy()
    mat = dfs['materials'].copy()
    hr = dfs['hr'].copy()
    cc = dfs['cost_centers'].copy()

    # Apply plant filter where possible
    if plant:
        mat = mat[mat['WERKS'].astype(str) == str(plant)]
        so_plant_matnrs = mat['MATNR'].unique()
        so = so[so['MATNR'].isin(so_plant_matnrs)]
        cc_for_plant = hr[hr['WERKS'].astype(str) == str(plant)]['KOSTL'].unique()
        cc = cc[cc['KOSTL'].isin(cc_for_plant)]
        logger.info('Plant filter applied: %s', plant)

    # Apply period filter (BW YYYYMM)
    if period:
        bw = bw[bw['CAL_YEAR_MONTH'] == int(period)]
        logger.info('Period filter applied: %s', period)

    kpis = {}

    # Revenue
    kpis['revenue'] = bw['REVENUE'].sum()
    kpis['gross_margin'] = bw['GROSS_MARGIN'].sum()
    kpis['gm_pct'] = (kpis['gross_margin'] / kpis['revenue'] * 100) if kpis['revenue'] > 0 else 0

    # Inventory
    mat['INV_VALUE'] = mat['LABST'] * mat['STPRS']
    kpis['inventory_value'] = mat['INV_VALUE'].sum()
    kpis['inventory_item_count'] = len(mat)

    # Open orders
    open_orders = so[so['STATUS'] == 'OPEN']
    kpis['open_order_value'] = open_orders['NETWR'].sum()
    kpis['open_order_count'] = len(open_orders)

    # Headcount
    active = hr[hr['TERM_DATE'].isna() | (pd.to_datetime(hr['TERM_DATE'], errors='coerce') > pd.Timestamp('today'))]
    kpis['active_headcount'] = len(active)
    kpis['avg_salary'] = active['SALARY'].mean() if len(active) > 0 else 0

    # Cost variance
    cc['VARIANCE'] = cc['ACTUAL_AMT'] - cc['PLAN_AMT']
    kpis['ytd_cost_variance'] = cc['VARIANCE'].sum()

    logger.info('KPIs calculated: %d metrics', len(kpis))
    return kpis

kpis = calculate_kpis(dfs, plant='1000', period=202412)
print('\nKPIs (plant=1000, period=202412):')
for k, v in kpis.items():
    print(f'  {k:<30} {v:,.2f}' if isinstance(v, float) else f'  {k:<30} {v}')

## 4. Module: format_report()

In [ ]:
# YOUR SOLUTION
import csv
from datetime import datetime

def format_report(
    kpis: dict,
    dq: pd.DataFrame,
    period: Optional[int] = None,
    output_dir: Path = Path('.')
) -> dict[str, Path]:
    """Format and export the analytics report to CSV and HTML.

    Args:
        kpis: KPI dict from calculate_kpis().
        dq: DQ scorecard DataFrame from run_dq_checks().
        period: Reporting period (YYYYMM) for file naming.
        output_dir: Directory to write output files.

    Returns:
        Dict mapping format name ('csv', 'html') to output file Path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    period_str = str(period) if period else 'ALL'
    ts = datetime.now().strftime('%Y%m%d_%H%M')

    # Build KPI summary DataFrame
    kpi_rows = [{'KPI': k.replace('_', ' ').title(), 'Value': v} for k, v in kpis.items()]
    kpi_df = pd.DataFrame(kpi_rows)

    # CSV export
    csv_path = output_dir / f'report_{period_str}_{ts}.csv'
    kpi_df.to_csv(csv_path, index=False)
    dq_path = output_dir / f'dq_scorecard_{period_str}_{ts}.csv'
    dq.to_csv(dq_path, index=False)
    logger.info('CSV reports written to %s', output_dir)

    # HTML export
    html_path = output_dir / f'report_{period_str}_{ts}.html'
    html_content = f"""<!DOCTYPE html>
<html><head><title>Analytics Report — Period {period_str}</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 40px; background: #f8f9fa; }}
  h1 {{ color: #2c3e50; }} h2 {{ color: #34495e; border-bottom: 2px solid #3498db; padding-bottom: 5px; }}
  table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; background: white; }}
  th {{ background: #3498db; color: white; padding: 8px 12px; text-align: left; }}
  td {{ padding: 8px 12px; border-bottom: 1px solid #ddd; }}
  tr:hover {{ background: #f1f8ff; }}
  .PASS {{ color: green; font-weight: bold; }}
  .FAIL {{ color: red; font-weight: bold; }}
  .WARN {{ color: orange; font-weight: bold; }}
</style></head><body>
<h1>Analytics Report — Period {period_str}</h1>
<p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
<h2>KPI Summary</h2>
{kpi_df.to_html(index=False, classes='kpi-table', border=0)}
<h2>Data Quality Scorecard</h2>
{dq.to_html(index=False, classes='dq-table', border=0)}
</body></html>"""

    html_path.write_text(html_content)
    logger.info('HTML report written to %s', html_path)

    return {'csv': csv_path, 'dq_csv': dq_path, 'html': html_path}

outputs = format_report(kpis, dq_scorecard, period=202412, output_dir=Path('.'))
print('\nReport outputs:')
for fmt, path in outputs.items():
    print(f'  {fmt}: {path}')

## 5. CLI Interface

In [ ]:
# YOUR SOLUTION
# In a real project, this would be in a separate file: report.py
# Here we simulate running it from within the notebook

cli_script = '''#!/usr/bin/env python3
"""
Analytics Report CLI

Usage:
    python report.py --plant 1000 --period 202412
    python report.py --period 202412
    python report.py  (run with all data)
"""
import argparse
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

def main():
    parser = argparse.ArgumentParser(description='Run analytics pipeline report.')
    parser.add_argument('--plant',  type=str, default=None, help='SAP Plant code (e.g., 1000)')
    parser.add_argument('--period', type=int, default=None, help='Reporting period YYYYMM (e.g., 202412)')
    parser.add_argument('--output', type=str, default='.', help='Output directory')
    args = parser.parse_args()

    logger.info('Starting pipeline: plant=%s, period=%s', args.plant, args.period)

    # In production, these functions would be imported from analytics_pipeline.py
    # For brevity, inline here (or import from the notebook module)
    data_path = Path(__file__).parent.parent / 'datasets'

    # dfs = load_data(data_path)
    # dq  = run_dq_checks(dfs)
    # kpis = calculate_kpis(dfs, plant=args.plant, period=args.period)
    # outputs = format_report(kpis, dq, period=args.period, output_dir=Path(args.output))

    # logger.info('Report complete: %s', outputs)
    print(f"[DEMO] Would run: plant={args.plant}, period={args.period}, output={args.output}")

if __name__ == '__main__':
    main()
'''

with open('report.py', 'w') as f:
    f.write(cli_script)
print('report.py written. Demo execution:')

import subprocess
result = subprocess.run(['python3', 'report.py', '--plant', '1000', '--period', '202412'],
                       capture_output=True, text=True)
print(result.stdout)

## 6. End-to-End Pipeline Run

In [ ]:
# YOUR SOLUTION
# Run the full pipeline as a single orchestrated call

def run_pipeline(plant: Optional[str] = None, period: Optional[int] = None) -> None:
    """Orchestrate the full analytics pipeline end-to-end."""
    logger.info('=== PIPELINE START: plant=%s period=%s ===', plant, period)

    try:
        dfs = load_data()
    except FileNotFoundError as e:
        logger.error('Data load failed: %s', e)
        return

    dq = run_dq_checks(dfs)
    fail_count = (dq['status'] == 'FAIL').sum()
    if fail_count > 0:
        logger.warning('%d DQ checks FAILED — proceeding with caution', fail_count)

    kpis = calculate_kpis(dfs, plant=plant, period=period)
    outputs = format_report(kpis, dq, period=period)

    logger.info('=== PIPELINE COMPLETE: %d KPIs, %d DQ checks, outputs: %s ===',
                len(kpis), len(dq), list(outputs.values()))

run_pipeline(plant='1000', period=202412)

## Daily Prompt
**Answer independently:**

1. The pipeline currently stops if `FileNotFoundError` is raised. Extend it: if a single CSV fails to load, log the error and continue with the remaining datasets (partial load mode). What trade-offs does this introduce for downstream KPI calculations?
2. Add a new function `send_report_summary(kpis: dict, recipient: str)` that prints (or would email) a 5-line text summary of the KPIs. Structure it to be easily swapped from print() to smtplib.
3. The CLI currently accepts `--plant` and `--period`. Add `--output-format` accepting 'csv', 'html', or 'both'. Update `format_report()` to respect this parameter.
